<a href="https://colab.research.google.com/github/bhpr0d/CCS8_SoftwareEng/blob/main/Copy_of_CCS8Githubrepo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from google.colab import files
from google.colab import drive
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Load data from KaggleHub
# The file name in the dataset 'uciml/breast-cancer-wisconsin-data' is 'data.csv'
file_path = 'data.csv'
try:
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "uciml/breast-cancer-wisconsin-data",
        file_path
    )
    print(f'Kaggle dataset loaded successfully from: uciml/breast-cancer-wisconsin-data/{file_path}')
except Exception as e:
    print(f"An error occurred while loading the Kaggle dataset: {e}")
    df = None

if df is not None:
    print('\nFirst 5 rows of the DataFrame:')
    display(df.head())

Using Colab cache for faster access to the 'breast-cancer-wisconsin-data' dataset.
Kaggle dataset loaded successfully from: uciml/breast-cancer-wisconsin-data/data.csv

First 5 rows of the DataFrame:


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


# XG BOOST

In [2]:
import pandas as pd
import xgboost as xgb
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss, precision_score, recall_score, f1_score, roc_auc_score
from google.colab import files

# DATA PREPARATION
# Assuming df is available from the previous cell

# Correctly define the target variable (y) as 'diagnosis'
y = df['diagnosis'].map({'M': 1, 'B': 0})  # Encode 'M' as 1 (Malignant) and 'B' as 0 (Benign)

# Correctly define the features (X) by dropping 'id', 'diagnosis', and 'Unnamed: 32'
# 'Unnamed: 32' is often an empty or redundant column in this dataset
X = df.drop(columns=['id', 'diagnosis', 'Unnamed: 32'], errors='ignore')

print(f"Target column (y): {y.name} (encoded: M=1, B=0)")
print(f"Features (X): {list(X.columns)}")

# Z-SCORE APPLICATION (For Picky Models)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# XGBoost Model Initialization and Training
# NOTE: Use scikit-learn wrapper
model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss')

# Modify model.fit to capture training metrics
model.fit(X_train, y_train, eval_set=[(X_train, y_train)], verbose=False)

# Make predictions on the test set
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate performance metrics for test set
accuracy = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_pred_proba)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Display performance metrics in a table (Test Set)
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Log Loss', 'Precision', 'Recall', 'F1-Score', 'ROC AUC'],
    'Value': [accuracy, loss, precision, recall, f1, roc_auc]
})

print("\nModel Performance on Test Set:")
display(metrics_df)

# Extract and display training logloss history
training_logloss = model.evals_result_['validation_0']['logloss']
training_metrics_df = pd.DataFrame({
    'Epoch': range(1, len(training_logloss) + 1),
    'Training Logloss': training_logloss
})

print('\nTraining Logloss History:')
display(training_metrics_df)

Target column (y): diagnosis (encoded: M=1, B=0)
Features (X): ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']

Model Performance on Test Set:


,Metric,Value
0,Accuracy,0.956140
1,Log Loss,0.122083
2,Precision,0.952381
3,Recall,0.930233
4,F1-Score,0.941176
5,ROC AUC,0.990829



Training Logloss History:


,Epoch,Training Logloss
0,1,0.441861
1,2,0.315433
2,3,0.235491
3,4,0.180223
4,5,0.141897
...,...,...
95,96,0.005457
96,97,0.005437
97,98,0.005415
98,99,0.005396
